# Contrastive Siamese Network for Sensor-Based Gesture Recognition

This notebook implements a **Supervised Contrastive Siamese Network** for the Kaggle challenge. 
It utilizes the temporal `SequenceExtractor` from `base_utils_qwen` to generate rich multi-domain features.

**Architecture Options:**
- **Backbone:** 1D CNN with dilated/standard convolutions.
- **Temporal Aggregation:** Bidirectional GRU (`gru`), Transformer Encoder (`attention`), or Global Pooling (`pool`).
- **Sequence Masking:** Automatically masks padded timesteps so temporal models don't process noise.

We provide both `GridSearchCV` and `BayesSearchCV` pipelines.

In [1]:
import sys
import os
import warnings
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
import random
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, train_test_split

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

try:
    dataset_name = os.listdir("/kaggle/input/datasets/keithmarange")[0]
    sys.path.append(f"/kaggle/input/datasets/keithmarange/{dataset_name}/")
    sys.path.append("/kaggle/input/cmi-competition-code")
except Exception:
    pass

import data_utils
from base_utils_qwen import (
    SequenceExtractor,
    competition_scorer,
    evaluate_holdout,
    make_competition_scorer,
    prepare_bayesian_space,
)
from utils_siamese_contrastive import KerasContrastiveSiameseClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer
from skopt.space import Categorical, Integer, Real
from base_utils_qwen import prepare_bayesian_space
import random

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "grid"  # "grid" or "bayesian"

random_state = 42
n_splits = 3
n_iter = 15
train_size = 0.7
error_score_constant = 0.0
verbose = 2
do_cross_val = False

# Fast local smoke test on data/eg.csv (set False for full train.csv)
use_eg_sample = False
eg_sample_pct = 0.02

# Optional row-level filters before split
do_handness = False
filter_non_bfrb_classes = False
filter_orientation_class_list = None  # e.g. ["Seated Straight"]
slice_by_orientation = None
slice_by_bfrb = None

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if target_col == "bfrb":
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

# Smaller training budget when using the eg.csv sample
default_epochs = 8 if use_eg_sample else 30
default_patience = 3 if use_eg_sample else 8
default_batch_size = 16 if use_eg_sample else 32

print(f"Search mode: {search_mode}")
print(f"use_eg_sample: {use_eg_sample}")

Search mode: grid
use_eg_sample: False


In [3]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    print(f"Using eg.csv sample: {raw_train_df['sequence_id'].nunique()} sequences")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(
            frac=eg_sample_pct, random_state=random_state
        )
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()
        print(f"Using {eg_sample_pct:.0%} sequence sample: {raw_train_df['sequence_id'].nunique()} sequences")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness and "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    train_df = train_df.drop(columns=["handedness"])

upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

if filter_non_bfrb_classes:
    train_df = train_df.loc[train_df["is_target"]].copy()

if filter_orientation_class_list is not None:
    train_df = train_df[train_df[orientation_col].isin(filter_orientation_class_list)].copy()

if slice_by_bfrb is True:
    train_df = train_df.loc[train_df["is_target"]].copy()
elif slice_by_bfrb is False:
    train_df = train_df.loc[~train_df["is_target"]].copy()

if slice_by_orientation is not None:
    train_df = train_df[train_df[orientation_col].isin(slice_by_orientation)].copy()

if filter_orientation_class_list is not None:
    sequences = train_df[["sequence_id", "is_target", target_col, orientation_col]].drop_duplicates()
    train_seqs, test_seqs = train_test_split(
        sequences["sequence_id"],
        test_size=(1 - train_size),
        stratify=sequences[target_col],
        random_state=random_state,
    )
    train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()
else:
    try:
        train_sample_df, hold_out_df = data_utils.sample_balanced_split(
            train_df,
            train_pct=train_size,
            test_pct=min(0.2, 1 - train_size),
            random_state=random_state,
        )
    except ValueError as exc:
        print(f"Balanced split unavailable ({exc}); using simple sequence split.")
        unique_seqs = (
            train_df[["sequence_id", target_col]]
            .drop_duplicates("sequence_id")
            .sample(frac=1, random_state=random_state)
        )
        n_train = max(1, int(len(unique_seqs) * train_size))
        train_seqs = unique_seqs.iloc[:n_train]["sequence_id"]
        test_seqs = unique_seqs.iloc[n_train:]["sequence_id"]
        if len(test_seqs) == 0 and len(unique_seqs) > 1:
            test_seqs = unique_seqs.iloc[-1:]["sequence_id"]
            train_seqs = unique_seqs.iloc[:-1]["sequence_id"]
        train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
        hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()
        print(
            f"Train: {train_sample_df['sequence_id'].nunique()} seqs | "
            f"Test: {hold_out_df['sequence_id'].nunique()} seqs"
        )

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique()}")
print(f"X_train rows: {len(X_train):,} | y_train rows: {len(y_train):,}")

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Train: 3879 seqs | 47.6%
Test:  966 seqs  | 11.9%
Train sequences: 3879 | Test sequences: 966
X_train rows: 280,205 | y_train rows: 280,205


In [4]:
# ============================================================
# FEATURE EXTRACTOR PARAMETERS - ALL OPTIONS
# ============================================================

ACC_MODES_OPTIONS = [
    "raw",
    "raw|velocity",
    "raw|velocity|jerk",
    "raw|velocity|displacement|jerk",
    "smoothed",
    "smoothed|velocity",
    "smoothed|velocity|jerk",
    "smoothed|velocity|displacement|jerk",
    "raw|smoothed",
    "raw|smoothed|velocity|jerk",
]

ROTATION_MODES_OPTIONS = [
    "quaternion",
    "euler",
    "quaternion|euler",
    "quaternion|angular_velocity",
    "euler|angular_velocity",
    "quaternion|euler|angular_velocity",
    "rot6d",
    "quaternion|rot6d",
    "euler|rot6d",
    "quaternion|euler|angular_velocity|rot6d",
]

TOF_MODES_OPTIONS = [
    "raw",
    "sensor_stats",
    "pooled_stats",
    "raw|sensor_stats",
    "sensor_stats|pooled_stats",
    "raw|pooled_stats",
    "raw|sensor_stats|pooled_stats",
    "pooled",
]

THM_MODES_OPTIONS = [
    "raw",
    "centered",
    "diff",
    "centered_diff",
    "raw|centered",
    "raw|diff",
    "centered|diff",
    "raw|centered_diff",
    "raw|centered|diff",
]

# ============================================================
# GRID SEARCH PARAMETER SPACE
# ============================================================

extractor_space = {
    "extractor__acc_modes": ACC_MODES_OPTIONS,
    "extractor__rotation_modes": ROTATION_MODES_OPTIONS,
    "extractor__tof_modes": TOF_MODES_OPTIONS,
    "extractor__thm_modes": THM_MODES_OPTIONS,
    "extractor__sampling_rate": [20, 100, 200],
    "extractor__maxlen": [128, 160, 200, 256],
    "extractor__window_size": [3, 5, 7, 10],
    "extractor__clip_value": [None],
    "extractor__interp_mode": ["linear", "ffill"],
    "extractor__motion_filter_mode": [None, "kalman"],
    "extractor__kalman_process_noise": [1e-4, 1e-3, 1e-2],
    "extractor__kalman_measurement_noise": [1e-2, 1e-1, 1.0],
    "extractor__padding_value": [0.0],
}

classifier_space = {
    # Siamese Network Parameters
    "classifier__target": ["bfrb"],
    "classifier__maxlen": [128, 160, 200, 256],
    "classifier__padding_value": [0.0],
    "classifier__backbone_filters": ["32-64", "64-128", "32-64-128", "64-128-256"],
    "classifier__kernel_sizes": ["3-3", "5-3", "3-5-3"],
    "classifier__temporal_mode": ["attention", "gru", "pool"],
    "classifier__embedding_dim": [32, 64, 128, 256],
    "classifier__contrastive_weight": [0.1, 0.3, 0.5, 0.7, 0.9],
    "classifier__temperature": [0.05, 0.1, 0.15, 0.2, 0.3],
    "classifier__dense_units": ["32", "64", "32-64", "64-32"],
    "classifier__dropout": [0.1, 0.2, 0.3, 0.4, 0.5],
    "classifier__learning_rate": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "classifier__batch_size": [16, 32, 64, 128],
    "classifier__epochs": [50],
    "classifier__patience": [10],
    "classifier__verbose": [0],
    "classifier__random_state": [42],
}

# ============================================================
# BAYESIAN PARAMETER SPACE (skopt)
# ============================================================

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    param_space = {
        # FEATURE EXTRACTOR
        "extractor__acc_modes": Categorical(ACC_MODES_OPTIONS),
        "extractor__rotation_modes": Categorical(ROTATION_MODES_OPTIONS),
        "extractor__tof_modes": Categorical(TOF_MODES_OPTIONS),
        "extractor__thm_modes": Categorical(THM_MODES_OPTIONS),
        "extractor__sampling_rate": Categorical([20, 50, 100, 200]),
        "extractor__maxlen": Integer(120, 320, prior="uniform"),
        "extractor__window_size": Integer(3, 15, prior="uniform"),
        "extractor__clip_value": Categorical([None]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical(['extended_kalman', "kalman"]),
        "extractor__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-3, 10.0, prior="log-uniform"),
        "extractor__padding_value": Categorical([0.0]),
        
        # SIAMESE NETWORK
        "classifier__target": Categorical([target_col]),
        "classifier__maxlen": Integer(120, 320, prior="uniform"),
        "classifier__padding_value": Categorical([0.0]),
        "classifier__backbone_filters": Categorical(["32-64", "64-128", "32-64-128", "64-128-256"]),
        "classifier__kernel_sizes": Categorical(["3-3", "5-3", "3-5-3"]),
        "classifier__temporal_mode": Categorical(["attention", "gru", "pool"]),
        "classifier__embedding_dim": Integer(32, 256, prior="uniform"),
        "classifier__contrastive_weight": Real(0.05, 0.95, prior="uniform"),
        "classifier__temperature": Real(0.01, 0.5, prior="log-uniform"),
        "classifier__dense_units": Categorical(["32", "64", "32-64", "64-32"]),
        "classifier__dropout": Real(0.05, 0.6, prior="uniform"),
        "classifier__learning_rate": Real(1e-5, 1e-1, prior="log-uniform"),
        "classifier__batch_size": Categorical([16, 32, 64, 128]),
        "classifier__epochs": Categorical([50]),
        "classifier__patience": Categorical([10]),
        "classifier__verbose": Categorical([0]),
        "classifier__random_state": Categorical([42]),
    }
    
    # Convert complex objects to JSON strings for skopt
    param_space = prepare_bayesian_space(param_space)
    
else:
    # GRID SEARCH MODE
    param_space = {**extractor_space, **classifier_space}

# ============================================================
# PRINT SUMMARY
# ============================================================

print("=" * 60)
print("PARAMETER SPACE SUMMARY - SIAMESE NETWORK")
print("=" * 60)
print(f"Search Mode: {search_mode}")
print(f"Total extractor parameters: {len(extractor_space)}")
print(f"Total classifier parameters: {len(classifier_space)}")

# Count total combinations (approx for grid search)
if search_mode != "bayesian":
    total_combos = 1
    for key, values in param_space.items():
        total_combos *= len(values)
    print(f"Total grid combinations: {total_combos:,}")
else:
    print("Bayesian mode: Continuous parameter sampling")

print("\n--- Feature Extractor Options ---")
print(f"  ACC_MODES: {len(ACC_MODES_OPTIONS)} options")
print(f"  ROTATION_MODES: {len(ROTATION_MODES_OPTIONS)} options")
print(f"  TOF_MODES: {len(TOF_MODES_OPTIONS)} options")
print(f"  THM_MODES: {len(THM_MODES_OPTIONS)} options")
print(f"  Sampling rates: {param_space['extractor__sampling_rate']}")
print(f"  Maxlen: {param_space['extractor__maxlen']}")
print(f"  Window size: {param_space['extractor__window_size']}")
print(f"  Motion filter: {param_space['extractor__motion_filter_mode']}")

print("\n--- Siamese Network Options ---")
print(f"  backbone_filters: {param_space['classifier__backbone_filters']}")
print(f"  kernel_sizes: {param_space['classifier__kernel_sizes']}")
print(f"  temporal_mode: {param_space['classifier__temporal_mode']}")
print(f"  embedding_dim: {param_space['classifier__embedding_dim']}")
print(f"  contrastive_weight: {param_space['classifier__contrastive_weight']}")
print(f"  temperature: {param_space['classifier__temperature']}")
print(f"  dropout: {param_space['classifier__dropout']}")
print(f"  learning_rate: {param_space['classifier__learning_rate']}")
print(f"  batch_size: {param_space['classifier__batch_size']}")

# ============================================================
# SAMPLE COMBINATION VISUALIZATION
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE CONFIGURATION (Random Combination)")
print("=" * 60)

sample_config = {}
for key, values in param_space.items():
    if hasattr(values, 'categories'):
        sample_config[key] = random.choice(values.categories)
    elif hasattr(values, 'low') or hasattr(values, 'high'):
        if isinstance(values, Integer):
            sample_config[key] = random.randint(values.low, values.high)
        else:
            sample_config[key] = f"CONTINUOUS [{values.low}, {values.high}]"
    else:
        sample_config[key] = random.choice(values) if isinstance(values, list) else values

for key, value in sample_config.items():
    print(f"  {key}: {value}")

print("\nParameter space ready for optimization.")

# ============================================================
# NEXT CELL: RUN GRID SEARCH
# ============================================================

PARAMETER SPACE SUMMARY - SIAMESE NETWORK
Search Mode: grid
Total extractor parameters: 13
Total classifier parameters: 17
Total grid combinations: 71,663,616,000,000

--- Feature Extractor Options ---
  ACC_MODES: 10 options
  ROTATION_MODES: 10 options
  TOF_MODES: 8 options
  THM_MODES: 9 options
  Sampling rates: [20, 100, 200]
  Maxlen: [128, 160, 200, 256]
  Window size: [3, 5, 7, 10]
  Motion filter: [None, 'kalman']

--- Siamese Network Options ---
  backbone_filters: ['32-64', '64-128', '32-64-128', '64-128-256']
  kernel_sizes: ['3-3', '5-3', '3-5-3']
  temporal_mode: ['attention', 'gru', 'pool']
  embedding_dim: [32, 64, 128, 256]
  contrastive_weight: [0.1, 0.3, 0.5, 0.7, 0.9]
  temperature: [0.05, 0.1, 0.15, 0.2, 0.3]
  dropout: [0.1, 0.2, 0.3, 0.4, 0.5]
  learning_rate: [0.0001, 0.0003, 0.001, 0.003, 0.01]
  batch_size: [16, 32, 64, 128]

SAMPLE CONFIGURATION (Random Combination)
  extractor__acc_modes: raw
  extractor__rotation_modes: rot6d
  extractor__tof_modes: sensor

In [5]:
# ============================================================
# EXTRACTOR CHANNEL PREVIEW
# ============================================================
sample_df = X_train.head(min(2000, len(X_train)))
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]
is_bayesian = search_mode == "bayesian" and SKOPT_AVAILABLE

if is_bayesian:

    print("Bayesian mode: sampling 3 extractor configs for channel preview\n")
    preview_combos = []
    for _ in range(3):
        params = {}
        for key in extractor_keys:
            param_name = key.replace("extractor__", "")
            space = param_space[key]
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                value = random.uniform(space.low, space.high)
            else:
                value = space
            params[param_name] = value
        params["padding_value"] = 0.0
        preview_combos.append(params)
else:
    extractor_values = [param_space[k] for k in extractor_keys]
    preview_combos = [
        {k.replace("extractor__", ""): v for k, v in zip(extractor_keys, combo)}
        for combo in itertools.islice(itertools.product(*extractor_values), 3)
    ]
    for params in preview_combos:
        params["padding_value"] = 0.0

for i, extractor_params in enumerate(preview_combos, start=1):
    try:
        extractor = SequenceExtractor(**extractor_params)
        extractor.fit(sample_df)
        out = extractor.transform(sample_df)
        n_channels = out["X"].shape[2]
        key_params = {k: extractor_params[k] for k in ["acc_modes", "rotation_modes", "maxlen", "sampling_rate"] if k in extractor_params}
        print(f"Preview {i}: {key_params} -> channels={n_channels}, timesteps={out['X'].shape[1]}")
    except Exception as exc:
        print(f"Preview {i} failed: {exc}")

Preview 1: {'acc_modes': 'raw', 'rotation_modes': 'quaternion', 'maxlen': 128, 'sampling_rate': 20} -> channels=996, timesteps=128
Preview 2: {'acc_modes': 'raw', 'rotation_modes': 'quaternion', 'maxlen': 128, 'sampling_rate': 20} -> channels=996, timesteps=128
Preview 3: {'acc_modes': 'raw', 'rotation_modes': 'quaternion', 'maxlen': 128, 'sampling_rate': 20} -> channels=996, timesteps=128


In [6]:
# ============================================================
# PIPELINE & SEARCH
# ============================================================
pipe = Pipeline([
    ("extractor", SequenceExtractor(padding_value=0.0)),
    (
        "classifier",
        KerasContrastiveSiameseClassifier(
            target=target_col,
            epochs=default_epochs,
            patience=default_patience,
            batch_size=default_batch_size,
            verbose=0,
            random_state=random_state,
        ),
    ),
])

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        pipe,
        param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        error_score=error_score_constant,
        verbose=verbose,
        return_train_score=True,
    )
else:
    search = GridSearchCV(
        pipe,
        param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        error_score=error_score_constant,
        verbose=verbose,
        return_train_score=True,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

Running grid search...


MemoryError: 

In [ ]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================
if not do_cross_val and len(X_test) > 0:
    y_pred = best_model.predict(X_test)
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")

    if target_col == "bfrb":
        eval_dict = evaluate_holdout(y_test_seq, y_pred, target_col=target_col, verbose=True)
        print(f"Holdout Competition Score: {eval_dict['competition_score']:.4f}")
    else:
        print("\nClassification Report:")
        print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))
        macro_f1 = f1_score(y_test_seq[target_col], y_pred, average="macro", zero_division=0)
        print(f"Macro F1 Score: {macro_f1:.4f}")
else:
    print("Holdout evaluation skipped (cross-val mode or empty test set).")